## Загрузка данных

---



In [1]:
!pip install --upgrade transformers

In [1]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
import numpy as np

In [2]:
from sklearn.metrics import precision_score, recall_score, f1_score

In [3]:
df = pd.read_csv('SMSSpamCollection',
                 sep='\t',
                 header=None,
                 names=['target','text'],
                 encoding='utf-8')

In [4]:
df

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [5]:
df['target'] = df['target'].map({'ham':0,'spam':1})

In [6]:
df

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...
5568,0,Will ü b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other s..."
5570,0,The guy did some bitching but I acted like i'd...


In [7]:
df['len_text'] = df['text'].apply(lambda x: len(x))

In [8]:
df.describe()

,target,len_text
count,5572.000000,5572.000000
mean,0.134063,80.489950
std,0.340751,59.942907
min,0.000000,2.000000
25%,0.000000,36.000000
50%,0.000000,62.000000
75%,0.000000,122.000000
max,1.000000,910.000000


In [9]:
X_train, X_test, y_train, y_test = train_test_split(df['text'].tolist(),
                                                    df['target'].tolist(),
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=df['target'])

## Датасет

In [10]:
class SMSDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['target'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Эксперимент с max_len

## max_len=64

### токенизатор

In [13]:
MAX_LEN = 64
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=MAX_LEN)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=MAX_LEN)
train_dataset = SMSDataset(train_encodings, y_train)
test_dataset = SMSDataset(test_encodings, y_test)
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

### Модель

In [14]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [16]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [16]:
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['target'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc

In [18]:
epochs = 3
best_val_acc = 0.0
log_interval = 50
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['target'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        if (step + 1) % log_interval == 0:
            current_loss = loss.item()
            print(f"Epoch {epoch+1}, Step {step+1}/{len(train_loader)}, Loss: {current_loss:.4f}")

    avg_train_loss = train_loss / len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader, device)

    print(f"Epoch {epoch+1} compl. | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pt')
        tokenizer.save_pretrained('./sms_spam_tokenizer')
        print(f"  -> Model saved. (acc={val_acc:.4f})")

Epoch 1, Step 50/279, Loss: 0.0309
Epoch 1, Step 100/279, Loss: 0.0574
Epoch 1, Step 150/279, Loss: 0.0047
Epoch 1, Step 200/279, Loss: 0.0093
Epoch 1, Step 250/279, Loss: 0.0038
Epoch 1 compl. | Train Loss: 0.0732 | Val Loss: 0.0410 | Val Acc: 0.9892
  -> Model saved. (acc=0.9892)
Epoch 2, Step 50/279, Loss: 0.0068
Epoch 2, Step 100/279, Loss: 0.0017
Epoch 2, Step 150/279, Loss: 0.0015
Epoch 2, Step 200/279, Loss: 0.0027
Epoch 2, Step 250/279, Loss: 0.0013
Epoch 2 compl. | Train Loss: 0.0180 | Val Loss: 0.0316 | Val Acc: 0.9919
  -> Model saved. (acc=0.9919)
Epoch 3, Step 50/279, Loss: 0.0082
Epoch 3, Step 100/279, Loss: 0.1198
Epoch 3, Step 150/279, Loss: 0.0005
Epoch 3, Step 200/279, Loss: 0.0010
Epoch 3, Step 250/279, Loss: 0.0011
Epoch 3 compl. | Train Loss: 0.0111 | Val Loss: 0.0344 | Val Acc: 0.9901


## max_len=128

### токенизатор

In [26]:
MAX_LEN = 128
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=MAX_LEN)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=MAX_LEN)
train_dataset = SMSDataset(train_encodings, y_train)
test_dataset = SMSDataset(test_encodings, y_test)
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

### Модель

In [27]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [28]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [29]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [30]:
epochs = 3
best_val_acc = 0.0
log_interval = 50
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['target'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        if (step + 1) % log_interval == 0:
            current_loss = loss.item()
            print(f"Epoch {epoch+1}, Step {step+1}/{len(train_loader)}, Loss: {current_loss:.4f}")

    avg_train_loss = train_loss / len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader, device)

    print(f"Epoch {epoch+1} compl. | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model_l128.pt')
        tokenizer.save_pretrained('./sms_spam_tokenizer_l128')
        print(f"  -> Model saved. (acc={val_acc:.4f})")

Epoch 1, Step 50/279, Loss: 0.0234
Epoch 1, Step 100/279, Loss: 0.0456
Epoch 1, Step 150/279, Loss: 0.0252
Epoch 1, Step 200/279, Loss: 0.2940
Epoch 1, Step 250/279, Loss: 0.0116
Epoch 1 compl. | Train Loss: 0.0792 | Val Loss: 0.0370 | Val Acc: 0.9928
  -> Model saved. (acc=0.9928)
Epoch 2, Step 50/279, Loss: 0.0012
Epoch 2, Step 100/279, Loss: 0.0010
Epoch 2, Step 150/279, Loss: 0.0045
Epoch 2, Step 200/279, Loss: 0.1678
Epoch 2, Step 250/279, Loss: 0.0029
Epoch 2 compl. | Train Loss: 0.0157 | Val Loss: 0.0415 | Val Acc: 0.9910
Epoch 3, Step 50/279, Loss: 0.0013
Epoch 3, Step 100/279, Loss: 0.0005
Epoch 3, Step 150/279, Loss: 0.0003
Epoch 3, Step 200/279, Loss: 0.0030
Epoch 3, Step 250/279, Loss: 0.0009
Epoch 3 compl. | Train Loss: 0.0153 | Val Loss: 0.0305 | Val Acc: 0.9928


## max_len=256

### токенизатор

In [31]:
MAX_LEN = 256
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=MAX_LEN)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=MAX_LEN)
train_dataset = SMSDataset(train_encodings, y_train)
test_dataset = SMSDataset(test_encodings, y_test)
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

### Модель

In [32]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [34]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [35]:
epochs = 3
best_val_acc = 0.0
log_interval = 50
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['target'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        if (step + 1) % log_interval == 0:
            current_loss = loss.item()
            print(f"Epoch {epoch+1}, Step {step+1}/{len(train_loader)}, Loss: {current_loss:.4f}")

    avg_train_loss = train_loss / len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader, device)

    print(f"Epoch {epoch+1} compl. | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model_l256.pt')
        tokenizer.save_pretrained('./sms_spam_tokenizer_l256')
        print(f"  -> Model saved. (acc={val_acc:.4f})")

Epoch 1, Step 50/279, Loss: 0.0329
Epoch 1, Step 100/279, Loss: 0.0509
Epoch 1, Step 150/279, Loss: 0.0064
Epoch 1, Step 200/279, Loss: 0.0160
Epoch 1, Step 250/279, Loss: 0.0025
Epoch 1 compl. | Train Loss: 0.0814 | Val Loss: 0.0366 | Val Acc: 0.9874
  -> Model saved. (acc=0.9874)
Epoch 2, Step 50/279, Loss: 0.0022
Epoch 2, Step 100/279, Loss: 0.0032
Epoch 2, Step 150/279, Loss: 0.0038
Epoch 2, Step 200/279, Loss: 0.0027
Epoch 2, Step 250/279, Loss: 0.2178
Epoch 2 compl. | Train Loss: 0.0182 | Val Loss: 0.0539 | Val Acc: 0.9857
Epoch 3, Step 50/279, Loss: 0.0034
Epoch 3, Step 100/279, Loss: 0.0008
Epoch 3, Step 150/279, Loss: 0.0032
Epoch 3, Step 200/279, Loss: 0.0014
Epoch 3, Step 250/279, Loss: 0.0015
Epoch 3 compl. | Train Loss: 0.0115 | Val Loss: 0.0356 | Val Acc: 0.9901
  -> Model saved. (acc=0.9901)


## оценка качества

In [17]:
def evaluate_all_metrics(model, dataloader, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['target'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    return acc, precision, recall, f1

In [38]:
output = []
for l in [64,128,256]:
  tokenizer = BertTokenizer.from_pretrained(f'./sms_spam_tokenizer_l{l}')
  test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=MAX_LEN)
  test_dataset = SMSDataset(test_encodings, y_test)
  batch_size = 16
  val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
  model.load_state_dict(torch.load(f'best_model_l{l}.pt', map_location=device))
  model.to(device)
  model.eval()
  ac, pr, rec, f1 = evaluate_all_metrics(model, val_loader, device)
  output.append(f'max_len={l}:')
  output.append(f'accuracy: {ac:.3f} | precision: {pr:.3f} | recall: {rec:.3f} | f1: {f1:.3f}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [39]:
for x in output:
  print(x)

max_len=64:
accuracy: 0.992 | precision: 0.993 | recall: 0.946 | f1: 0.969
max_len=128:
accuracy: 0.993 | precision: 0.980 | recall: 0.966 | f1: 0.973
max_len=256:
accuracy: 0.990 | precision: 0.973 | recall: 0.953 | f1: 0.963


# Эксперимент с learning_rate

## learning_rate=2e-5

### токенизатор

In [11]:
MAX_LEN = 128
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=MAX_LEN)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=MAX_LEN)
train_dataset = SMSDataset(train_encodings, y_train)
test_dataset = SMSDataset(test_encodings, y_test)
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### Модель

In [12]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [14]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [18]:
epochs = 3
best_val_acc = 0.0
log_interval = 50
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['target'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        if (step + 1) % log_interval == 0:
            current_loss = loss.item()
            print(f"Epoch {epoch+1}, Step {step+1}/{len(train_loader)}, Loss: {current_loss:.4f}")

    avg_train_loss = train_loss / len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader, device)

    print(f"Epoch {epoch+1} compl. | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model_lr2.pt')
        tokenizer.save_pretrained('./sms_spam_tokenizer_lr2')
        print(f"  -> Model saved. (acc={val_acc:.4f})")

Epoch 1, Step 50/279, Loss: 0.0026
Epoch 1, Step 100/279, Loss: 0.0675
Epoch 1, Step 150/279, Loss: 0.0022
Epoch 1, Step 200/279, Loss: 0.0011
Epoch 1, Step 250/279, Loss: 0.0028
Epoch 1 compl. | Train Loss: 0.0179 | Val Loss: 0.0429 | Val Acc: 0.9892
  -> Model saved. (acc=0.9892)
Epoch 2, Step 50/279, Loss: 0.0005
Epoch 2, Step 100/279, Loss: 0.0004
Epoch 2, Step 150/279, Loss: 0.0004
Epoch 2, Step 200/279, Loss: 0.0005
Epoch 2, Step 250/279, Loss: 0.0021
Epoch 2 compl. | Train Loss: 0.0044 | Val Loss: 0.0458 | Val Acc: 0.9919
  -> Model saved. (acc=0.9919)
Epoch 3, Step 50/279, Loss: 0.0005
Epoch 3, Step 100/279, Loss: 0.0002
Epoch 3, Step 150/279, Loss: 0.0001
Epoch 3, Step 200/279, Loss: 0.0002
Epoch 3, Step 250/279, Loss: 0.0001
Epoch 3 compl. | Train Loss: 0.0007 | Val Loss: 0.0640 | Val Acc: 0.9910


## learning_rate=3e-5

### токенизатор

In [19]:
MAX_LEN = 128
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=MAX_LEN)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=MAX_LEN)
train_dataset = SMSDataset(train_encodings, y_train)
test_dataset = SMSDataset(test_encodings, y_test)
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

### Модель

In [20]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [22]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

In [23]:
epochs = 3
best_val_acc = 0.0
log_interval = 50
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['target'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        if (step + 1) % log_interval == 0:
            current_loss = loss.item()
            print(f"Epoch {epoch+1}, Step {step+1}/{len(train_loader)}, Loss: {current_loss:.4f}")

    avg_train_loss = train_loss / len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader, device)

    print(f"Epoch {epoch+1} compl. | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model_lr3.pt')
        tokenizer.save_pretrained('./sms_spam_tokenizer_lr3')
        print(f"  -> Model saved. (acc={val_acc:.4f})")

Epoch 1, Step 50/279, Loss: 0.0146
Epoch 1, Step 100/279, Loss: 0.0553
Epoch 1, Step 150/279, Loss: 0.0108
Epoch 1, Step 200/279, Loss: 0.0054
Epoch 1, Step 250/279, Loss: 0.0073
Epoch 1 compl. | Train Loss: 0.0684 | Val Loss: 0.0440 | Val Acc: 0.9892
  -> Model saved. (acc=0.9892)
Epoch 2, Step 50/279, Loss: 0.0017
Epoch 2, Step 100/279, Loss: 0.0023
Epoch 2, Step 150/279, Loss: 0.0017
Epoch 2, Step 200/279, Loss: 0.1688
Epoch 2, Step 250/279, Loss: 0.0064
Epoch 2 compl. | Train Loss: 0.0199 | Val Loss: 0.0296 | Val Acc: 0.9937
  -> Model saved. (acc=0.9937)
Epoch 3, Step 50/279, Loss: 0.0007
Epoch 3, Step 100/279, Loss: 0.0006
Epoch 3, Step 150/279, Loss: 0.0010
Epoch 3, Step 200/279, Loss: 0.0010
Epoch 3, Step 250/279, Loss: 0.0005
Epoch 3 compl. | Train Loss: 0.0055 | Val Loss: 0.0460 | Val Acc: 0.9919


## learning_rate=5e-5

### токенизатор

In [24]:
MAX_LEN = 128
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=MAX_LEN)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=MAX_LEN)
train_dataset = SMSDataset(train_encodings, y_train)
test_dataset = SMSDataset(test_encodings, y_test)
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

### Модель

In [25]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [27]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

In [28]:
epochs = 3
best_val_acc = 0.0
log_interval = 50
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['target'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        if (step + 1) % log_interval == 0:
            current_loss = loss.item()
            print(f"Epoch {epoch+1}, Step {step+1}/{len(train_loader)}, Loss: {current_loss:.4f}")

    avg_train_loss = train_loss / len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader, device)

    print(f"Epoch {epoch+1} compl. | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model_lr5.pt')
        tokenizer.save_pretrained('./sms_spam_tokenizer_lr5')
        print(f"  -> Model saved. (acc={val_acc:.4f})")

Epoch 1, Step 50/279, Loss: 0.0225
Epoch 1, Step 100/279, Loss: 0.0424
Epoch 1, Step 150/279, Loss: 0.3134
Epoch 1, Step 200/279, Loss: 0.0949
Epoch 1, Step 250/279, Loss: 0.0049
Epoch 1 compl. | Train Loss: 0.0806 | Val Loss: 0.0454 | Val Acc: 0.9919
  -> Model saved. (acc=0.9919)
Epoch 2, Step 50/279, Loss: 0.0016
Epoch 2, Step 100/279, Loss: 0.0026
Epoch 2, Step 150/279, Loss: 0.0020
Epoch 2, Step 200/279, Loss: 0.0015
Epoch 2, Step 250/279, Loss: 0.0014
Epoch 2 compl. | Train Loss: 0.0267 | Val Loss: 0.0318 | Val Acc: 0.9910
Epoch 3, Step 50/279, Loss: 0.0006
Epoch 3, Step 100/279, Loss: 0.0004
Epoch 3, Step 150/279, Loss: 0.0005
Epoch 3, Step 200/279, Loss: 0.0004
Epoch 3, Step 250/279, Loss: 0.0009
Epoch 3 compl. | Train Loss: 0.0062 | Val Loss: 0.0383 | Val Acc: 0.9928
  -> Model saved. (acc=0.9928)


## оценка качества

In [29]:
def evaluate_all_metrics(model, dataloader, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['target'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    return acc, precision, recall, f1

In [30]:
output = []
for lr in [2,3,5]:
  tokenizer = BertTokenizer.from_pretrained(f'./sms_spam_tokenizer_lr{lr}')
  test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=MAX_LEN)
  test_dataset = SMSDataset(test_encodings, y_test)
  batch_size = 16
  val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
  model.load_state_dict(torch.load(f'best_model_lr{lr}.pt', map_location=device))
  model.to(device)
  model.eval()
  ac, pr, rec, f1 = evaluate_all_metrics(model, val_loader, device)
  output.append(f'learning_rate={lr}:')
  output.append(f'accuracy: {ac:.3f} | precision: {pr:.3f} | recall: {rec:.3f} | f1: {f1:.3f}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [31]:
for x in output:
  print(x)

learning_rate=2:
accuracy: 0.992 | precision: 0.986 | recall: 0.953 | f1: 0.969
learning_rate=3:
accuracy: 0.994 | precision: 0.986 | recall: 0.966 | f1: 0.976
learning_rate=5:
accuracy: 0.993 | precision: 0.980 | recall: 0.966 | f1: 0.973


# Weighted classes

In [35]:
from sklearn.utils import compute_class_weight

In [37]:
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

In [39]:
crit = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

In [48]:
MAX_LEN = 128
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=MAX_LEN)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=MAX_LEN)
train_dataset = SMSDataset(train_encodings, y_train)
test_dataset = SMSDataset(test_encodings, y_test)
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [49]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [50]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [51]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [52]:
def evaluate_weighted(model, dataloader, crit, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['target'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = crit(outputs.logits, labels)
            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc

epochs = 3
best_val_acc = 0.0
log_interval = 50
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['target'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = crit(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        if (step + 1) % log_interval == 0:
            current_loss = loss.item()
            print(f"Epoch {epoch+1}, Step {step+1}/{len(train_loader)}, Loss: {current_loss:.4f}")

    avg_train_loss = train_loss / len(train_loader)
    val_loss, val_acc = evaluate_weighted(model, val_loader, crit, device)

    print(f"Epoch {epoch+1} compl. | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model_weighted.pt')
        tokenizer.save_pretrained('./sms_spam_tokenizer_weighted')
        print(f"  -> Model saved. (acc={val_acc:.4f})")

Epoch 1, Step 50/279, Loss: 0.0283
Epoch 1, Step 100/279, Loss: 0.0185
Epoch 1, Step 150/279, Loss: 0.1396
Epoch 1, Step 200/279, Loss: 0.0104
Epoch 1, Step 250/279, Loss: 0.0093
Epoch 1 compl. | Train Loss: 0.1346 | Val Loss: 0.0728 | Val Acc: 0.9731
  -> Model saved. (acc=0.9731)
Epoch 2, Step 50/279, Loss: 0.0022
Epoch 2, Step 100/279, Loss: 0.1517
Epoch 2, Step 150/279, Loss: 0.0103
Epoch 2, Step 200/279, Loss: 0.0055
Epoch 2, Step 250/279, Loss: 0.0056
Epoch 2 compl. | Train Loss: 0.0421 | Val Loss: 0.0681 | Val Acc: 0.9830
  -> Model saved. (acc=0.9830)
Epoch 3, Step 50/279, Loss: 0.0009
Epoch 3, Step 100/279, Loss: 0.0008
Epoch 3, Step 150/279, Loss: 0.0011
Epoch 3, Step 200/279, Loss: 0.0007
Epoch 3, Step 250/279, Loss: 0.0011
Epoch 3 compl. | Train Loss: 0.0156 | Val Loss: 0.0734 | Val Acc: 0.9857
  -> Model saved. (acc=0.9857)


## сравнение

In [53]:
output = []

tokenizer = BertTokenizer.from_pretrained(f'./sms_spam_tokenizer_lr3')
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=MAX_LEN)
test_dataset = SMSDataset(test_encodings, y_test)
batch_size = 16
val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.load_state_dict(torch.load(f'best_model_lr3.pt', map_location=device))
model.to(device)
model.eval()
ac, pr, rec, f1 = evaluate_all_metrics(model, val_loader, device)
output.append(f'unweighted:')
output.append(f'accuracy: {ac:.3f} | precision: {pr:.3f} | recall: {rec:.3f} | f1: {f1:.3f}')

tokenizer = BertTokenizer.from_pretrained(f'./sms_spam_tokenizer_weighted')
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=MAX_LEN)
test_dataset = SMSDataset(test_encodings, y_test)
batch_size = 16
val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.load_state_dict(torch.load(f'best_model_weighted.pt', map_location=device))
model.to(device)
model.eval()
ac, pr, rec, f1 = evaluate_all_metrics(model, val_loader, device)
output.append(f'weighted:')
output.append(f'accuracy: {ac:.3f} | precision: {pr:.3f} | recall: {rec:.3f} | f1: {f1:.3f}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [54]:
for x in output:
  print(x)

unweighted:
accuracy: 0.994 | precision: 0.986 | recall: 0.966 | f1: 0.976
weighted:
accuracy: 0.986 | precision: 0.935 | recall: 0.960 | f1: 0.947
